# Stream 流式输出

## Overview 概述

在 AI 开发中，**Stream（流式输出）** 不仅仅是为了让前端显示那个“打字机效果”，它更是为了让开发者看清：**在一个复杂的图中，数据是如何一跳一跳（Hop by Hop）流动的。**

LangChain 的流式传输系统允许你将 Agent 运行过程中的实时反馈直接呈现给你的应用程序。 **通过 LangChain Streaming 你可以实现：**

- **流式传输 Agent 进度**：在 Agent 的每一步操作（节点执行）后获取状态更新。
- **流式传输 LLM Token**：在语言模型生成字符的同时，实时获取这些 Token。
- **流式传输思考/推理过程**：实时呈现模型的推理链（思维链）内容。
- **流式传输自定义更新**：发出用户定义的信号（例如：“已获取 10/100 条记录”）。
- **流式传输多种模式**：支持 `updates`（Agent 进度）、`messages`（LLM Token + 元数据）或 `custom`（任意自定义数据）。

在之前的学习里，你用 `agent.invoke` 就像是发快递：你把包裹投进去，然后坐在门口等，直到包裹送到目的地（模型跑完所有逻辑），你才能拿到回执。

**Streaming（流式传输）** 则像是快递轨迹追踪：你可以实时看到包裹到了哪个分拣中心，分拣员说了什么，甚至能看到包裹里正在打印的文字。



#### **核心逻辑一：`updates`（进度流）—— 谁在干活？**

这是最基础的流。每当你的 LangGraph 走完一个节点（比如从 `agent` 走到 `tools`），它就会吐出一个 `chunk`。

- **Webster 实战意义**：如果你写了一个复杂的 Agent，调了三个工具。通过 `updates`，你可以在前端显示：“正在查询数据库...”、“正在分析数据...”。用户就不会觉得程序卡死了。

#### **核心逻辑二：`messages`（Token 流）—— 逐字蹦出**

这是你最熟悉的“打字机效果”。

- **Webster 实战意义**：本地模型（比如 Qwen 7B）在处理复杂问题时，生成完整回复可能需要 5-10 秒。如果你用 `invoke`，用户要干等 10 秒；用 `messages` 流，用户在第 0.5 秒就能看到第一个字。这种**感知速度**的提升是巨大的。

#### **核心逻辑三：思维链流（Reasoning Tokens）—— 模型在想什么？**

现在的模型（比如 DeepSeek-R1 或开启思维链的 Qwen）在回答之前会有一段“思考”。

- **Webster 实战意义**：以前这些思考过程是隐藏的。现在你可以把模型那段 `thought` 标签里的内容实时流式传出来。这让你的 Webster 看起来非常有“灵性”，像是在现场打草稿。

#### **核心逻辑四：`custom`（自定义信号）—— 汇报进度**

这是给开发者留的“后门”。

- **Webster 实战意义**：假设你在写一个爬虫工具。你可以手动在代码里发一个信号：“正在抓取第 5 个网页...”。这个信号会随着流一起传到前端，而不需要干扰正常的对话逻辑。

| **模式 (Mode)** | **它吐出什么？**        | **最适合干什么？**                          |
| --------------- | ----------------------- | ------------------------------------------- |
| **`values`**    | 整个 State 的当前镜像   | 调试！看每个节点跑完后 State 变成了什么样。 |
| **`updates`**   | 仅限该节点产生的变化    | 逻辑追踪。看哪个节点被触发了。              |
| **`messages`**  | 原始的 Token 和消息碎片 | **前端 UI 必备**。做打字机效果和感知优化。  |

## Stream Mode 流模式

将以下一种或多种流模式作为列表传递给[`stream`](https://reference.langchain.com/python/langgraph/graphs/#langgraph.graph.state.CompiledStateGraph.stream)or[`astream`](https://reference.langchain.com/python/langgraph/graphs/#langgraph.graph.state.CompiledStateGraph.astream)方法：

| 模式       | 描述                                                         |
| :--------- | :----------------------------------------------------------- |
| `updates`  | 代理程序每执行一步操作后，都会传输状态更新。如果在同一步骤中进行多次更新（例如，运行多个节点），则这些更新会分别传输。 |
| `messages` | `(token, metadata)`从调用 LLM 的任何图节点获取元组流。       |
| `custom`   | 使用流写入器从图节点内部流式传输自定义数据。                 |
